# 确定性波阻抗反演

这个 notebook 按 `ginn` 当前的数据入口组织确定性反演流程，但不再做全体积全时窗求解，而是只在“目标层上下加边界样点”的时间窗内反演。

当前版本采用：

- `forward` 差分的 post-stack 正演算子
- `epsR` 空间正则
- `damp` 对背景模型的阻尼收缩
- 井仅用于结果验收
- 导出 `npy`、`npz` 和 `SEG-Y`


In [ ]:
import json
import logging
import sys
import time
from pathlib import Path

import cigsegy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pylops import Laplacian
from pylops.avo.poststack import PoststackLinearModelling
from pylops.optimization.leastsquares import regularized_inversion

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import import_interpretation_petrel, import_seismic, import_well_heads_petrel
from cup.seismic.target_layer import TargetLayer
from cup.seismic.survey import open_survey
from ginn.config import GINNConfig
from ginn.data import estimate_wavelet_gain, load_lowfreq_model, make_lowfreq_model, make_wavelet

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans SC", "SimSun"]
plt.rcParams["axes.unicode_minus"] = False
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

cfg = GINNConfig.from_yaml(repo_root / "experiments/ginn/train.yaml", base_dir=repo_root)
cfg.lfm_source = "wtie_time_lfm"

output_dir = repo_root / "data" / "output_deterministic_inversion_20260418"
output_dir.mkdir(parents=True, exist_ok=True)

params = {
    "boundary_effect_samples": 30,
    "epsR": 0.20,
    "damp": 0.03,
    "iter_lim": 100,
    "show": True,
}

print("Repo root:", repo_root)
print("Output dir:", output_dir)
print(json.dumps(params, indent=2, ensure_ascii=False))

In [ ]:
def sample_axis_seconds(geometry):
    n_sample = int(geometry["n_sample"])
    sample_min = float(geometry["sample_min"])
    sample_step = float(geometry["sample_step"])
    axis = sample_min + np.arange(n_sample, dtype=np.float64) * sample_step
    sample_unit = str(geometry.get("sample_unit", "")).strip().lower()
    if sample_unit in {"s", "sec", "secs", "second", "seconds"}:
        return axis
    if sample_unit in {"ms", "msec", "millisecond", "milliseconds"}:
        return axis / 1000.0
    if np.nanmax(np.abs(axis)) > 100.0:
        return axis / 1000.0
    return axis


def build_textual_header(title, lines):
    rows = [f"C{idx:>2d} {text}"[:80].ljust(80) for idx, text in enumerate([title, *lines], start=1)]
    rows.extend([f"C{idx:>2d}".ljust(80) for idx in range(len(rows) + 1, 41)])
    textual = "".join(rows)
    if len(textual) != 3200:
        raise ValueError(f"Expected 3200-char textual header, got {len(textual)}")
    return textual


def bilinear_trace_from_volume(volume, survey_ctx, x, y):
    i_float, j_float = survey_ctx.coord_to_index(x, y)
    i0 = int(np.floor(i_float))
    i1 = int(np.ceil(i_float))
    j0 = int(np.floor(j_float))
    j1 = int(np.ceil(j_float))
    i1 = min(i1, volume.shape[0] - 1)
    j1 = min(j1, volume.shape[1] - 1)
    wi = float(i_float - i0)
    wj = float(j_float - j0)
    t00 = volume[i0, j0, :]
    t01 = volume[i0, j1, :]
    t10 = volume[i1, j0, :]
    t11 = volume[i1, j1, :]
    return (1.0 - wi) * (1.0 - wj) * t00 + (1.0 - wi) * wj * t01 + wi * (1.0 - wj) * t10 + wi * wj * t11


def collect_wtie_references(ai_dir):
    refs = {}
    for csv_path in sorted(Path(ai_dir).glob("*.csv")):
        stem = csv_path.stem
        if "_" not in stem:
            continue
        well_name, _ = stem.rsplit("_", 1)
        refs[well_name.upper()] = csv_path
    return refs


print("加载地震体、几何、层位和背景模型...")
seismic = import_seismic(
    cfg.seismic_file,
    seismic_type="segy",
    iline=cfg.segy_iline,
    xline=cfg.segy_xline,
    istep=cfg.segy_istep,
    xstep=cfg.segy_xstep,
)

survey_ctx = open_survey(
    cfg.seismic_file,
    seismic_type="segy",
    segy_options={
        "iline": cfg.segy_iline,
        "xline": cfg.segy_xline,
        "istep": cfg.segy_istep,
        "xstep": cfg.segy_xstep,
    },
)
geometry = survey_ctx.query_geometry(domain="time")
sample_axis_s = sample_axis_seconds(geometry)

top_df_raw = import_interpretation_petrel(cfg.top_horizon_file)
bot_df_raw = import_interpretation_petrel(cfg.bot_horizon_file)
target_layer = TargetLayer(
    raw_horizon_dfs={"top": top_df_raw, "bottom": bot_df_raw},
    geometry=geometry,
    horizon_names=["top", "bottom"],
    outlier_threshold=0.02,
    outlier_min_neighbor_count=2,
)
mask = target_layer.to_mask()
lfm = load_lowfreq_model(cfg.precomputed_lfm_file)  # type: ignore

if seismic.shape != mask.shape or seismic.shape != lfm.shape:
    raise ValueError(f"Shape mismatch: seismic={seismic.shape}, mask={mask.shape}, lfm={lfm.shape}")

target_samples = mask.any(axis=(0, 1))
if not np.any(target_samples):
    raise ValueError("Target layer mask is empty.")

target_start = int(np.argmax(target_samples))
target_end = int(len(target_samples) - np.argmax(target_samples[::-1]))  # type: ignore
window_start = max(0, target_start - int(params["boundary_effect_samples"]))
window_end = min(mask.shape[-1], target_end + int(params["boundary_effect_samples"]))

seismic_window = seismic[:, :, window_start:window_end].astype(np.float32)
lfm_window = np.clip(lfm[:, :, window_start:window_end], a_min=1e-6, a_max=None).astype(np.float32)
mask_window = mask[:, :, window_start:window_end]
sample_axis_s_window = sample_axis_s[window_start:window_end]

valid_window = mask_window.any(axis=2)
if not np.any(valid_window):
    raise ValueError("Windowed target layer has no valid traces.")

window_valid_values = seismic_window[mask_window.astype(bool)]
seis_rms = float(np.sqrt(np.mean(window_valid_values**2))) + 1e-10
candidate_trace_indices = np.flatnonzero(mask.reshape(-1, mask.shape[-1]).any(axis=1))

if cfg.wavelet_gain is None:
    unit_wavelet = make_wavelet(
        wavelet_type=cfg.wavelet_type,
        freq=cfg.wavelet_freq,
        dt=cfg.wavelet_dt,
        length=cfg.wavelet_length,
        gain=1.0,
    )
    wavelet_gain = estimate_wavelet_gain(
        seismic,
        lfm,
        mask,
        unit_wavelet,
        seis_rms=seis_rms,
        max_traces=cfg.wavelet_gain_num_traces,
        candidate_trace_indices=candidate_trace_indices,
    )
else:
    wavelet_gain = float(cfg.wavelet_gain)

wavelet = make_wavelet(
    wavelet_type=cfg.wavelet_type,
    freq=cfg.wavelet_freq,
    dt=cfg.wavelet_dt,
    length=cfg.wavelet_length,
    gain=wavelet_gain,
)

d_obs = np.moveaxis(seismic_window / seis_rms, -1, 0).astype(np.float32)
m0_log = np.moveaxis(np.log(lfm_window), -1, 0).astype(np.float32)
nt_window, n_il, n_xl = d_obs.shape

print("原始体 shape:", seismic.shape)
print("目标层样点区间:", target_start, target_end)
print("带边界效应缓冲后的反演区间:", window_start, window_end)
print("窗口反演 shape (nt, nil, nxl):", d_obs.shape)
print("seis_rms:", seis_rms)
print("wavelet_gain:", wavelet_gain)

In [ ]:
print("构建正演和空间正则算子...")
Pop = PoststackLinearModelling(
    wavelet,
    nt0=nt_window,
    spatdims=(n_il, n_xl),
    explicit=False,
    kind="forward",
)
Reg_spatial = Laplacian((nt_window, n_il, n_xl), axes=(1, 2), dtype=np.float32)

print("开始反演...")
t0 = time.time()
xinv_log, istop, itn, r1norm, r2norm = regularized_inversion(
    Pop,
    d_obs.ravel(),
    [Reg_spatial],
    x0=m0_log.ravel(),
    epsRs=[params["epsR"]],
    show=params["show"],
    iter_lim=params["iter_lim"],
    damp=params["damp"],
    atol=1e-4,
    btol=1e-4,
)
elapsed = time.time() - t0

inv_log_window = xinv_log.reshape(d_obs.shape)
inv_ai_window = np.moveaxis(np.exp(inv_log_window), 0, -1).astype(np.float32)
syn_norm_window = (Pop * xinv_log).reshape(d_obs.shape)
residual_norm_window = d_obs - syn_norm_window

full_ai = lfm.copy().astype(np.float32)
full_ai[:, :, window_start:window_end] = inv_ai_window

summary = {
    "elapsed_sec": elapsed,
    "istop": int(istop),
    "itn": int(itn),
    "r1norm": float(r1norm),
    "r2norm": float(r2norm),
    "window_start": int(window_start),
    "window_end": int(window_end),
    "ai_min": float(inv_ai_window.min()),
    "ai_max": float(inv_ai_window.max()),
    "ai_mean": float(inv_ai_window.mean()),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
ai_full_npy_path = output_dir / "deterministic_ai_full.npy"
meta_npz_path = output_dir / "deterministic_inversion_meta.npz"
summary_json_path = output_dir / "run_summary.json"
segy_path = output_dir / "deterministic_ai_full.segy"
lfm_for_ginn_segy = output_dir / "deterministic_lfm_for_ginn.segy"

np.save(ai_full_npy_path, full_ai)
np.savez(
    meta_npz_path,
    sample_axis_s=sample_axis_s.astype(np.float32),
    sample_axis_s_window=sample_axis_s_window.astype(np.float32),
    window_start=np.int32(window_start),
    window_end=np.int32(window_end),
    mask=mask.astype(np.uint8),
    geometry_json=json.dumps(geometry, ensure_ascii=False),
    params_json=json.dumps(params, ensure_ascii=False),
)
with summary_json_path.open("w", encoding="utf-8") as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2)

cigsegy.create_by_sharing_header(
    str(segy_path),
    str(cfg.seismic_file),
    np.ascontiguousarray(full_ai.astype(np.float32)),
    keylocs=[cfg.segy_iline, cfg.segy_xline, cfg.segy_istep, cfg.segy_xstep],
    textual=build_textual_header(
        "Deterministic impedance volume",
        [
            f"window={window_start}:{window_end}",
            f"epsR={params['epsR']}",
            f"damp={params['damp']}",
            f"kind=forward iter_lim={params['iter_lim']}",
        ],
    ),
)

lfm_for_ginn = make_lowfreq_model(
    full_ai,
    dt_s=cfg.dt,
    cutoff_hz=cfg.lfm_cutoff_hz,
    order=cfg.lfm_filter_order,
)
cigsegy.create_by_sharing_header(
    str(lfm_for_ginn_segy),
    str(cfg.seismic_file),
    np.ascontiguousarray(lfm_for_ginn.astype(np.float32)),
    keylocs=[cfg.segy_iline, cfg.segy_xline, cfg.segy_istep, cfg.segy_xstep],
    textual=build_textual_header(
        "Low-frequency model from deterministic inversion",
        [
            f"cutoff_hz={cfg.lfm_cutoff_hz}",
            f"order={cfg.lfm_filter_order}",
        ],
    ),
)

mid_il = full_ai.shape[0] // 2
shared_vmin = float(np.nanpercentile(np.concatenate([lfm[mid_il], full_ai[mid_il]], axis=None), 2))
shared_vmax = float(np.nanpercentile(np.concatenate([lfm[mid_il], full_ai[mid_il]], axis=None), 98))
diff = full_ai[mid_il] - lfm[mid_il]
diff_abs = float(np.nanpercentile(np.abs(diff), 98))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
im0 = axes[0].imshow(
    full_ai[mid_il].T, cmap="viridis", aspect="auto", origin="upper", vmin=shared_vmin, vmax=shared_vmax
)
axes[0].set_title(f"反演 AI | inline index={mid_il}")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(lfm[mid_il].T, cmap="viridis", aspect="auto", origin="upper", vmin=shared_vmin, vmax=shared_vmax)
axes[1].set_title(f"背景 LFM | inline index={mid_il}")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

im2 = axes[2].imshow(diff.T, cmap="seismic", aspect="auto", origin="upper", vmin=-diff_abs, vmax=diff_abs)
axes[2].set_title(f"反演 AI - LFM | inline index={mid_il}")
fig.colorbar(im2, ax=axes[2], shrink=0.85)

figure_path = output_dir / f"inline_{mid_il:04d}_deterministic_vs_lfm.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

print("已保存完整反演体:", ai_full_npy_path)
print("已保存反演 SEG-Y:", segy_path)
print("已保存低通 LFM SEG-Y:", lfm_for_ginn_segy)
print("已保存图件:", figure_path)

In [ ]:
print("执行井验收...")
well_heads = import_well_heads_petrel(repo_root / "data/raw/well_heads")
well_heads = well_heads.assign(Name_norm=well_heads["Name"].str.upper())
reference_files = collect_wtie_references(repo_root / "data/output_vertical_well_wtie_20260416/ai")

metrics = []
example_plot_done = False

for well_name_norm, csv_path in reference_files.items():
    head_match = well_heads.loc[well_heads["Name_norm"] == well_name_norm]
    if head_match.empty:
        print(f"跳过 {well_name_norm}: 井头未匹配")
        continue

    head = head_match.iloc[0]
    ref_df = pd.read_csv(csv_path)
    ref_df = ref_df.rename(columns={c: c.strip().lower() for c in ref_df.columns})
    if "twt_s" not in ref_df.columns or "ai" not in ref_df.columns:
        print(f"跳过 {well_name_norm}: 列名异常 {list(ref_df.columns)}")
        continue

    pred_trace = bilinear_trace_from_volume(full_ai, survey_ctx, float(head["Surface X"]), float(head["Surface Y"]))
    pred_ai = np.interp(ref_df["twt_s"].to_numpy(), sample_axis_s, pred_trace, left=np.nan, right=np.nan)
    ref_ai = ref_df["ai"].to_numpy(dtype=np.float64)
    valid = np.isfinite(pred_ai) & np.isfinite(ref_ai)
    if not np.any(valid):
        print(f"跳过 {well_name_norm}: 无重叠采样")
        continue

    diff = pred_ai[valid] - ref_ai[valid]
    mae = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))
    bias = float(np.mean(diff))
    corr = float(np.corrcoef(pred_ai[valid], ref_ai[valid])[0, 1]) if np.sum(valid) > 1 else np.nan

    metrics.append(
        {
            "well_name": head["Name"],
            "n_samples": int(np.sum(valid)),
            "mae": mae,
            "rmse": rmse,
            "bias": bias,
            "corr": corr,
            "reference_file": csv_path.name,
        }
    )

    if not example_plot_done:
        fig, ax = plt.subplots(figsize=(5, 10))
        ax.plot(ref_ai[valid], ref_df.loc[valid, "twt_s"], label="WTIE AI", lw=2)
        ax.plot(pred_ai[valid], ref_df.loc[valid, "twt_s"], label="确定性反演 AI", lw=2)
        ax.invert_yaxis()
        ax.set_xlabel("AI")
        ax.set_ylabel("TWT (s)")
        ax.set_title(f"井验收 | {head['Name']}")
        ax.legend(loc="best")
        ax.grid(True, alpha=0.3, linestyle=":")
        qc_plot_path = output_dir / f"well_qc_{str(head['Name']).replace('/', '_')}.png"
        fig.savefig(qc_plot_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("已保存井验收图:", qc_plot_path)
        example_plot_done = True

metrics_df = pd.DataFrame(metrics)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values("rmse")
metrics_csv_path = output_dir / "well_qc_metrics.csv"
metrics_df.to_csv(metrics_csv_path, index=False)
print("已保存井验收指标:", metrics_csv_path)
metrics_df